# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

## Setup

Import the required libraries:

| Import | Purpose |
|---|---|
| `openai.OpenAI` | Python SDK used for **both** OpenAI and Ollama — Ollama exposes an OpenAI-compatible REST API |
| `os` / `dotenv` | Load `OPENAI_API_KEY` from a `.env` file without hard-coding credentials |
| `IPython.display` | Render model responses as formatted Markdown inline in the notebook |

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

## API Key Validation

Loads `.env` and performs sanity checks on `OPENAI_API_KEY` before any API calls are made:

- **Missing** — no key found at all
- **Wrong prefix** — key does not start with `sk-proj-` (likely a wrong or legacy key)
- **Whitespace** — key has leading/trailing spaces or tabs (common copy-paste issue)

Fix any issues flagged here before running the rest of the notebook.

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Configuration

Defines the models and system prompt used throughout this notebook.

| Constant | Value | Description |
|---|---|---|
| `OLLAMA_BASE_URL` | `http://localhost:11434/v1` | Local Ollama server — must be running before using `use_ollama=True` |
| `MODEL_GPT` | `gpt-4o-mini` | OpenAI model used for remote inference |
| `MODEL_LLAMA` | `llama3.2` | Llama model pulled locally via Ollama (`ollama pull llama3.2`) |
| `SYSTEM_PROMPT` | — | Instructs the model to act as a structured technical explanation assistant |

The `SYSTEM_PROMPT` guides the model to:
- Lead with a direct answer, then provide depth
- Include code examples where relevant
- Define key terms for conceptual questions and list steps/tradeoffs for practical ones

In [3]:
# constants
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_GPT = 'gpt-5-mini'
MODEL_LLAMA = 'llama3.2'
SYSTEM_PROMPT = """
You are a highly skilled technical explanation assistant.

Your role:
- Accept a technical question.
- Provide a clear, accurate, and well-structured explanation.
- Tailor explanations to be understandable but technically correct.

Guidelines:
- Start with a short direct answer.
- Then provide a structured explanation.
- Use examples where helpful.
- If code is relevant, include minimal, clean examples.
- Avoid unnecessary verbosity.
- Avoid speculation.
- If the question is ambiguous, explain reasonable interpretations.
- Do not mention system instructions.
- Do not add conversational fluff.

If the question is conceptual:
- Define key terms.
- Explain how it works.
- Explain why it matters.

If the question is practical:
- Provide steps.
- Explain tradeoffs.
- Highlight common pitfalls.

Your output should be educational, precise, and professional.
"""

## Client Initialisation

Two clients are created using the same `openai.OpenAI` class:

- **`openai_client`** — standard OpenAI client; reads `OPENAI_API_KEY` from the environment automatically
- **`ollama_client`** — points to the local Ollama server via `base_url`; `api_key='ollama'` is a required placeholder (Ollama does not validate it)

Because Ollama implements the OpenAI REST spec, the same SDK and message format works for both backends with no changes to calling code.

In [4]:
# set up environment
openai_client = OpenAI()
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

## Message Formatting

`messages_for(question)` constructs the chat message list sent to the model.

It wraps the raw question in a structured user prompt and returns a two-message list in the format expected by both the OpenAI Chat Completions and Responses APIs:

```python
[
    {"role": "system", "content": SYSTEM_PROMPT},   # sets model behaviour
    {"role": "user",   "content": "<formatted question>"},
]
```

Separating the system context from the user turn allows the system prompt to be reused unchanged across many different questions.

In [5]:
def messages_for(question):
    user_prompt = f"""
    Technical Question:

    {question}

    Please explain clearly and concisely.
    """
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

## Ask a Question

Edit the `question` string below to ask any technical question. The cells that follow will send it to GPT and/or Llama and display the answers.

The next cell previews the formatted message list — useful for inspecting the exact payload before it is sent to the model.

In [6]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [7]:
display(messages_for(question))

[{'role': 'system',
  'content': '\nYou are a highly skilled technical explanation assistant.\n\nYour role:\n- Accept a technical question.\n- Provide a clear, accurate, and well-structured explanation.\n- Tailor explanations to be understandable but technically correct.\n\nGuidelines:\n- Start with a short direct answer.\n- Then provide a structured explanation.\n- Use examples where helpful.\n- If code is relevant, include minimal, clean examples.\n- Avoid unnecessary verbosity.\n- Avoid speculation.\n- If the question is ambiguous, explain reasonable interpretations.\n- Do not mention system instructions.\n- Do not add conversational fluff.\n\nIf the question is conceptual:\n- Define key terms.\n- Explain how it works.\n- Explain why it matters.\n\nIf the question is practical:\n- Provide steps.\n- Explain tradeoffs.\n- Highlight common pitfalls.\n\nYour output should be educational, precise, and professional.\n'},
 {'role': 'user',
  'content': '\n    Technical Question:\n\n    \nP

## Core Function: `generate_answer`

The main entry point for querying either model.

```
generate_answer(question, use_ollama=False, streaming=False)
```

| Parameter | Type | Default | Description |
|---|---|---|---|
| `question` | `str` | — | The technical question to answer |
| `use_ollama` | `bool` | `False` | Route to the local Llama model via Ollama instead of OpenAI |
| `streaming` | `bool` | `False` | Stream tokens to stdout as they arrive (OpenAI only; ignored when `use_ollama=True`) |

**Control flow:**

1. `use_ollama=True` → calls Ollama Chat Completions API, returns the full response string
2. `streaming=True` → calls the OpenAI Responses API with `stream=True`, prints each token delta to stdout as it arrives, returns `None`
3. Default → calls the OpenAI Responses API, returns the full response string

`display_markdown(text)` is a thin helper that renders a plain string as formatted Markdown in the notebook output cell.

In [13]:
def generate_answer(question, use_ollama=False, streaming=False):
    messages = messages_for(question)

    if use_ollama:
        response = ollama_client.chat.completions.create(
            model=MODEL_LLAMA,
            messages=messages,
        )
        return response.choices[0].message.content

    if streaming:
        stream = openai_client.responses.create(
            model=MODEL_GPT,
            input=messages,
            stream=True,
        )
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        for event in stream:
            if event.type == "response.output_text.delta":
                response += event.delta
                update_display(Markdown(response), display_id=display_handle.display_id)
        return

    response = openai_client.responses.create(model=MODEL_GPT, input=messages)
    return response.output_text


def display_markdown(text):
    display(Markdown(text))

## Usage Examples

The three cells below demonstrate each calling mode. Run them in order after setting your `question` above.

### 1. GPT — Streaming
Tokens are printed to stdout incrementally as they arrive. Use this when you want to see the response build up in real time. Returns `None`.

### 2. GPT — Non-Streaming
Waits for the full response, then renders it as formatted Markdown. Cleaner output for reading; slightly higher latency before anything appears.

### 3. Llama 3.2 via Ollama (Local)
Runs entirely on your machine — no API key or internet required. Requires Ollama to be running (`ollama serve`) and the model to be pulled (`ollama pull llama3.2`). Response quality and speed depend on local hardware.

In [14]:
# Get gpt-5-mini to answer, with streaming
generate_answer(question, streaming=True)

Short answer
This expression yields each unique, truthy author value found in books (a collection of dict-like objects). It builds a set of non‑falsy authors and then yields each element of that set (in arbitrary order).

Explanation
- Components:
  - {book.get("author") for book in books if book.get("author")} is a set comprehension. For each book in books it calls book.get("author"); if that returned value is truthy, it is included in the set. The set removes duplicates.
  - yield from <iterable> iterates over the iterable and yields each item from the surrounding generator.

- What the code does (equivalent expanded form):
  - It constructs a set of authors that are truthy:
      authors = {book.get("author") for book in books if book.get("author")}
  - Then it yields each author from that set:
      for author in authors:
          yield author

- Important details and tradeoffs:
  - Duplicate removal: using a set removes duplicate authors.
  - Order: sets are unordered, so the yielded authors come in arbitrary order. If you need a deterministic or original order, use an ordered deduplication method.
  - Memory and latency: the set comprehension evaluates eagerly — all authors are collected before the first yield. For very large books, this uses more memory and delays the first yield.
  - Filtering: the if book.get("author") test excludes falsy values (None, empty string, 0, False). If you only want to exclude None but keep empty strings, change the test accordingly.
  - Minor inefficiency: book.get("author") is called twice per book in the comprehension. You can avoid that with the walrus operator (Python 3.8+):
      {author for book in books if (author := book.get("author"))}

Examples / alternatives
- If you want streaming (no eager set) but no deduplication:
    yield from (book.get("author") for book in books if book.get("author"))

- If you want deduplication while preserving first-seen order:
    seen = set()
    for book in books:
        author = book.get("author")
        if author and author not in seen:
            seen.add(author)
            yield author

- If you need to include empty-string authors but exclude None:
    yield from {book.get("author") for book in books if book.get("author") is not None}

In [10]:
# Get gpt-5-mini to answer, without streaming
answer = generate_answer(question, streaming=False)
display_markdown(answer)

Short answer
It yields each unique, truthy author value found in the iterable books. The set comprehension collects non-falsy authors and deduplicates them, and yield from iterates that set, yielding each author to the caller.

Explanation (step‑by‑step)
1. {book.get("author") for book in books if book.get("author")}
   - This is a set comprehension. For each book in books it calls book.get("author").
   - The if book.get("author") filters out falsy values (None, "", 0, etc.).
   - The set stores each remaining author exactly once (duplicates removed).

2. yield from <set>
   - In a generator function, yield from <iterable> iterates over the iterable and yields each element to the caller.
   - So yield from the set yields each unique author value.

Important consequences and caveats
- Deduplication: authors are unique because of the set.
- Order: sets are unordered, so the authors are yielded in arbitrary order (not the input order).
- Memory: the set is built fully in memory before yielding; for large inputs this uses memory proportional to the number of unique authors.
- Double evaluation: book.get("author") is called twice per book (once in the value expression, once in the if). If get has side effects or is expensive, that matters.
- Filtering removes all falsy values — if empty string is a legitimate author, it will be removed.

Better alternatives
- Preserve input order and deduplicate:
  def gen_authors(books):
      seen = set()
      for book in books:
          author = book.get("author")
          if author and author not in seen:
              seen.add(author)
              yield author

- If you just want to yield authors (including duplicates) lazily:
  yield from (book.get("author") for book in books if book.get("author"))

Example usage
def gen_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

books = [{"author":"A"}, {"author":"B"}, {"author":"A"}, {"title":"untitled"}, {"author":""}]
list(gen_authors(books))  # e.g. ['A', 'B'] (order not guaranteed)

Summary
The code yields unique, non-falsy author values from books, but builds a set (deduplicates and loses order) and calls book.get("author") twice per item. Consider alternatives if you need stable ordering, memory efficiency, or to avoid double evaluation.

In [15]:
question = "How can I configure django for sending email in production environment?"
generate_answer(question, streaming=True)

Short answer
Configure Django’s EMAIL_BACKEND and provider-specific settings in settings.py (using environment variables or a secrets manager), choose a reliable mail provider (SMTP or an API-based provider via django-anymail), enable TLS/SSL, send emails from a verified domain, and send mails asynchronously in production (task queue) while monitoring delivery and handling bounces.

Explanation and steps

1) Key Django settings
- EMAIL_BACKEND: which backend to use.
  - SMTP: "django.core.mail.backends.smtp.EmailBackend"
  - API providers (recommended): use django-anymail backends that call provider APIs.
- Core settings you will set in production:
  - EMAIL_HOST, EMAIL_PORT, EMAIL_HOST_USER, EMAIL_HOST_PASSWORD
  - EMAIL_USE_TLS or EMAIL_USE_SSL
  - DEFAULT_FROM_EMAIL and SERVER_EMAIL
  - EMAIL_TIMEOUT (seconds)
Example minimal SMTP settings (use env vars; do NOT commit secrets):


    import os
    EMAIL_BACKEND = "django.core.mail.backends.smtp.EmailBackend"
    EMAIL_HOST = os.environ.get("EMAIL_HOST")           # e.g. smtp.sendgrid.net
    EMAIL_PORT = int(os.environ.get("EMAIL_PORT", 587)) # 587 for STARTTLS, 465 for SSL
    EMAIL_HOST_USER = os.environ.get("EMAIL_HOST_USER")
    EMAIL_HOST_PASSWORD = os.environ.get("EMAIL_HOST_PASSWORD")
    EMAIL_USE_TLS = os.environ.get("EMAIL_USE_TLS", "True") == "True"
    EMAIL_USE_SSL = os.environ.get("EMAIL_USE_SSL", "False") == "True"
    EMAIL_TIMEOUT = 10
    DEFAULT_FROM_EMAIL = "Your App <noreply@example.com>"
    SERVER_EMAIL = DEFAULT_FROM_EMAIL

2) Use an API-based provider (recommended)
- Providers: SendGrid, Mailgun, Amazon SES, Postmark, SparkPost. They have higher deliverability, analytics, bounce handling, and rate limits.
- Use django-anymail to integrate easily with Django’s EmailMessage API and get provider-specific features.

Example with django-anymail + SendGrid (pip install django-anymail):

    INSTALLED_APPS += ["anymail"]
    EMAIL_BACKEND = "anymail.backends.sendgrid.EmailBackend"
    ANYMAIL = {
      "SENDGRID_API_KEY": os.environ.get("SENDGRID_API_KEY"),
    }
    DEFAULT_FROM_EMAIL = "noreply@example.com"

3) Sending emails in code
- Simple send_mail:

    from django.core.mail import send_mail
    send_mail("Subject", "Body text", "from@example.com", ["to@example.com"], fail_silently=False)

- For HTML and attachments use EmailMultiAlternatives:

    from django.core.mail import EmailMultiAlternatives
    msg = EmailMultiAlternatives("Subject", "Text body", "from@example.com", ["to@example.com"])
    msg.attach_alternative("<p>HTML body</p>", "text/html")
    msg.send()

4) Asynchronous sending (production best practice)
- Don’t block web requests on network I/O. Use a background worker: Celery, RQ, Huey, or cloud workers.
- Example pattern with Celery:

    @shared_task
    def send_email_task(subject, body, from_email, to):
        send_mail(subject, body, from_email, to)

    # in your view
    send_email_task.delay(subject, body, from_email, [to])

5) Security and configuration best practices
- Store credentials in environment variables or a secrets manager (AWS Secrets Manager, Vault).
- Use TLS/SSL; prefer STARTTLS on port 587 or SSL on 465 depending on provider.
- Verify sending domains with the provider (SPF, DKIM). Configure DMARC for improved deliverability.
- Use a dedicated sending domain or subdomain (e.g. mail.example.com) and add DNS records: SPF TXT, DKIM public key, and set MX if required.
- Limit DEFAULT_FROM_EMAIL to a domain you control and registered in provider console.

6) Error handling, rate limits, and monitoring
- Handle exceptions (SMTPAuthenticationError, smtplib.SMTPException).
- Use provider dashboards / webhooks to process bounces, complaints, unsubscribes. Many providers deliver bounce notifications to an endpoint.
- Respect rate limits. For bulk sends use provider batch APIs or throttle sends via task queue.
- Enable logging for django.core.mail.backends and provider responses.

7) Testing and staging
- Don’t send real emails from local dev. Use:
  - django.core.mail.backends.console.EmailBackend (prints to stdout)
  - file backend to write to files
  - Mailtrap or provider sandbox for safe testing
- Use separate API keys / sending domains for staging/production.

8) Common pitfalls
- Using Gmail SMTP for production: limited quotas, stricter security (use OAuth or app passwords), not suitable for high volume.
- Leaving EMAIL_BACKEND as console or file in production.
- Hard-coding credentials in repo.
- Not verifying DNS records => emails land in spam.
- Sending synchronously and blocking web requests.

9) Example: production-ready checklist
- Choose provider (SendGrid/Mailgun/SES/Postmark).
- Set EMAIL_BACKEND and provider credentials from environment.
- Verify sending domain and configure SPF/DKIM/DMARC.
- Use background tasks for sending.
- Implement bounce/complaint webhook handlers.
- Monitor delivery, logs, and quotas.

That’s the core. If you tell me which provider you plan to use (SendGrid, Mailgun, SES, Gmail, etc.), I can give a concrete settings.py example, anymail config, and webhook handling snippet.

In [11]:
# Get Llama 3.2 to answer
answer = generate_answer(question, use_ollama=True)
display_markdown(answer)

**Direct Answer:**

This code uses a generator expression to yield the author of each book in the `books` list where the author exists.

**Structured Explanation:**

The code is written in Python and utilizes several features:

1.  **List Comprehension**: The `{book.get("author") for book in books if book.get("author")}` part uses a list comprehension, which similar to a traditional for loop but more concise.

    ```python
for book in books:
    author = book.get("author")
    # Use the author (assuming it can be used elsewhere)
```

2.  **Yield from**: The `yield from` keyword is used to delegate a subgroup of items to a sub-iteration, allowing it to behave like one single iteration.

    ```python
def main():
    yield from {book.get("author") for book in books if book.get("author")}
```

Here's what happens: For each book in the `books` list, the code tries to retrieve the author. If the "author" key does not exist in this book dictionary (i.e., `book.get("author")` is `None`), it will stop iterating over that iteration and move on with another iteration.

    ```python
# Without yield from

authors = []
for book in books:
    if book.get("author"):
        authors.append(book.get("author"))
```

However, using `yield from` allows us to do something even more concise while making the code easier to read:

    ```python
# With yield from

def return_authors():
    for author in {book.get("author") for book in books if book.get("author"):
        # Use the author (assuming it can be used elsewhere)
```

    In essence, `yield from` makes our function a "subset iterator", allowing us to easily handle complex subsets of our `books)`.

This code can improve readability and memory efficiency at the same time. 

For simplicity, note that without using `yield from`, one would still have to return all authors, even if some books (`book`) in `books` do not contain author information:

   ```python
    # Return all authors only
for book in books:
    result = {author: book.get("author")}
    yield {
        key: value for (key, value) in result.items()
    }
```

In [16]:
answer = generate_answer(question, use_ollama=True)
display_markdown(answer)

**Configuring Django for Email Sending**

To send emails using Django in a production environment, follow these steps:

**Step 1: Install Required Packages**

Install `django mail` package by running:
```bash
pip install django-mail
```
In your `settings.py` file, add `'django_mail'` package to `INSTALLED_APPS`.

```python
# settings.py

INSTALLED_APPS = [
    # ...
    'django_mail',
]
```
**Step 2: Set Email Configuration**

 Configure your email server settings:

### Gmail
Use `smtp.gmail.com` as the mail server.

```python
# settings.py (email section)

EMAIL_BACKEND='django.core.mail.backends.smtp.EmailBackend'
EMAIL_HOST='smtp.gmail.com'
EMAIL_PORT=587
EMAIL_USE_TLS=False
EMAIL_HOST_USER='your_email@gmail.com'
EMAIL_HOST_PASSWORD='your_password'
```

### Other Mail Servers

Replace `smtp.gmail.com` with your preferred mail server, and configure the `EMAIL_PORT`, `EMAIL_USE_TLS`, and email address (`EMAIL_HOST_USER`) settings accordingly.

**Step 3: Test Email Sending (optional)**

Run a simple test to ensure email sending works:
```python
# tests.py

from django.core.mail import send_mail

send_mail('Test Email', 'Email from Django', 'your_email@gmail.com', ['recipient@example.com'])
```

That's it! With these configurations, you should now be able to send emails using Django in your production environment.

**Note:** If you're still having issues with email sending, refer to [official documentation](https://docs.djangoproject.com/en/4.0/ref/settings/#email-backend) for more detailed configuration options and troubleshooting guides.